# Run RCC on a Colab GPU

Two sender contexts feed one receiver, with two requests. This exercises
`bind`, `transfer` and native Hugging Face generation on Qwen3-8B. Optionally,
it then validates the vLLM adapter on each pinned vLLM version.

1. Select an **A100** (preferred) or **L4**, with a **high-RAM host**, under
   **Runtime > Change runtime type**. Unquantized Qwen3-8B needs a GPU with at
   least 24 GB-class memory; a T4 is too small. vLLM validation needs an A100.
2. To validate vLLM, set `VLLM_PINS` in the setup cell.
3. Run the setup cell and upload the source ZIP when prompted.
4. Allow Drive mounting, then run the remaining cells. Results are saved under
   `MyDrive/rcc-api/` in a new directory for each run.

Build the source ZIP from a committed checkout; uncommitted changes are excluded:

```bash
mkdir -p output/colab
git archive --format=zip --output=output/colab/rcc-api-colab-source.zip HEAD
```

This is a GPU smoke test, not a speed benchmark.

In [ ]:
import hashlib
import json
import logging
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch

MODEL = "Qwen/Qwen3-8B"
REVISION = None  # None pins the current commit; set a recorded hash to repeat a run.
VLLM_PINS: tuple[str, ...] = ()  # Add "0.11.1" and/or "0.26.0" to validate vLLM.
RATIO = 4
MAX_NEW_TOKENS = 128
TEST_MODE = os.environ.get("RCC_API_TEST_MODE") == "1"
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

if TEST_MODE:
    MODEL = os.environ["RCC_API_MODEL"]
    SOURCE_ZIP = Path(os.environ["RCC_API_SOURCE_ZIP"])
    OUTPUT_ROOT = Path(os.environ["RCC_API_OUTPUT"])
    DEVICE = "cpu"
    VLLM_PINS = ()
else:
    from google.colab import drive, files
    from huggingface_hub import HfApi

    REVISION = REVISION or HfApi(token=False).model_info(MODEL).sha

    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime before recording.")
    gpu = torch.cuda.get_device_properties(0)
    if gpu.total_memory < 22 * 1024**3:
        raise RuntimeError("Use a 24 GB-class GPU or larger for unquantized Qwen3-8B.")
    logging.info(
        "GPU: %s (%.1f GiB). Use a high-RAM host for model loading.",
        gpu.name,
        gpu.total_memory / 1024**3,
    )
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one RCC source ZIP.")
    SOURCE_ZIP = Path(next(iter(uploaded)))
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/rcc-api")
    DEVICE = "cuda"

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RUN = OUTPUT_ROOT / run_id
RUN.mkdir(parents=True)
SOURCE = Path(tempfile.mkdtemp(prefix="rcc-api-source-"))
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in archive.namelist()):
        raise ValueError("Source ZIP contains an unsafe path.")
    archive.extractall(SOURCE)
if not (SOURCE / "src/rcc/hf.py").is_file() or not (SOURCE / "pyproject.toml").is_file():
    raise ValueError("Expected a git archive of the RCC repository root.")
shutil.copyfile(SOURCE_ZIP, RUN / "source.zip")
(RUN / "config.json").write_text(
    json.dumps(
        {
            "model": MODEL,
            "revision": REVISION,
            "device": DEVICE,
            "ratio": RATIO,
            "max_new_tokens": MAX_NEW_TOKENS,
            "source_sha256": hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest(),
            "test_mode": TEST_MODE,
            "vllm_pins": list(VLLM_PINS),
        },
        indent=2,
    )
    + "\n"
)
logging.info("Run directory: %s", RUN)

## Install

Pins Transformers 4.57.1 and keeps Colab's PyTorch. Inference runs in a fresh
subprocess so no kernel restart is needed. The exact source ZIP, package
versions, GPU details and model revision are saved with the results.


In [ ]:
run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(SOURCE / "src") + os.pathsep + run_env.get("PYTHONPATH", "")
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env["HF_HUB_DISABLE_XET"] = "1"
if TEST_MODE:
    run_env["HF_HUB_OFFLINE"] = "1"


def run_command(command: list[str], log_name: str, env: dict[str, str] | None = None) -> None:
    """Save complete stdout/stderr and fail visibly when a command fails."""
    log_path = RUN / log_name
    logging.info("Running %s; log: %s", command, log_path)
    with log_path.open("a") as log:
        result = subprocess.run(
            command,
            cwd=SOURCE,
            env=run_env if env is None else env,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=False,
        )
    if result.returncode:
        logging.error("%s", log_path.read_text()[-4000:])
        raise RuntimeError(f"Command failed ({result.returncode}); see {log_path}")


if not TEST_MODE:
    run_command(
        [sys.executable, "-m", "pip", "install", str(SOURCE), "transformers==4.57.1"], "install.log"
    )
    run_command(["nvidia-smi"], "gpu.txt")
run_command([sys.executable, "-m", "pip", "freeze"], "packages.txt")

## Transfer and continue

The API path being exercised is:

```python
senders = [bind(model, tokenizer, prompt=document, backend="hf") for document in documents]
receiver = bind(model, tokenizer, backend="hf", max_new_tokens=128)
await transfer(senders, receiver, requests, ratio=4)
tokens = model.generate(**receiver.pop())
```

One delivery is queued per request. Each contains selected rows from both
senders. Generation stays in the receiver's own loop. The default selector
uses W=16, with no additional latent rollout, matching the API defaults.

Timings synchronize CUDA and separate model loading, sender prefill, transfer,
and receiver prefill plus generation. They are single-run observations, not a
comparison against text communication. The expected-text check is a simple
substring check, not a benchmark score. Inspect the raw answers, including errors.
Each completed answer is saved immediately; download partial logs if a run fails.


In [ ]:
script = r"""import asyncio
import json
import sys
import time
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

from rcc import bind, transfer

run = Path(sys.argv[1])
config = json.loads((run / "config.json").read_text())
device = config["device"]
torch.set_num_threads(4)
dtype = torch.float32 if device == "cpu" else (
    torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)
started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(config["model"], revision=config["revision"])
model = AutoModelForCausalLM.from_pretrained(
    config["model"], revision=config["revision"], dtype=dtype,
    attn_implementation="sdpa",
).to(device).eval()
if device == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
load_seconds = time.perf_counter() - started


def clock():
    if device == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter()


def save():
    if device == "cuda":
        result["peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
    (run / "results.json").write_text(json.dumps(result, indent=2) + "\n")


documents = [
    "Cedar is owned by Maya Chen and launches on October 12. " * 8,
    "Birch is owned by Luis Ortega and launches on November 8. " * 8,
]
requests = ["Who owns Cedar?", "When does Birch launch?"]
result = {
    "complete": False, "config": config, "torch": torch.__version__,
    "transformers": transformers.__version__, "dtype": str(dtype),
    "resolved_model_revision": getattr(model.config, "_commit_hash", None),
    "gpu": torch.cuda.get_device_name() if device == "cuda" else None,
    "model_load_seconds": load_seconds, "documents": documents, "cases": [],
}
save()
started = clock()
senders = [bind(model, tokenizer, prompt=document, backend="hf") for document in documents]
result["sender_prefill_seconds"] = clock() - started
result["sender_positions"] = [agent.sender_state().input_embeds.shape[0] for agent in senders]
receiver = bind(model, tokenizer, backend="hf", max_new_tokens=config["max_new_tokens"])
started = clock()
asyncio.run(transfer(senders, receiver, requests, ratio=config["ratio"]))
result["transfer_seconds"] = clock() - started
assert len(receiver) == len(requests)
assert result["sender_positions"] == [
    s.sender_state().past_key_values.get_seq_length() for s in senders
]
save()
for request, expected in zip(requests, ["Maya Chen", "November 8"], strict=True):
    started = clock()
    inputs = receiver.pop(discard_pending=True)
    preparation_seconds = clock() - started
    started = clock()
    tokens = model.generate(
        **inputs, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    seconds = clock() - started
    text = tokenizer.decode(tokens[0], skip_special_tokens=True)
    result["cases"].append({
        "request": request, "answer": text, "token_ids": tokens[0].tolist(),
        "receiver_input_positions": inputs["inputs_embeds"].shape[1],
        "input_preparation_seconds": preparation_seconds,
        "receiver_prefill_and_generation_seconds": seconds,
        "expected_text": expected, "contains_expected_text": expected.lower() in text.lower(),
    })
    save()
    print(f"{request}\n{text}\nReceiver: {seconds:.3f}s", flush=True)
assert not receiver
result["complete"] = True
save()
print(f"Saved {run / 'results.json'}", flush=True)
"""
(RUN / "run.py").write_text(script)
run_command([sys.executable, "-u", str(RUN / "run.py"), str(RUN)], "run.log")
result = json.loads((RUN / "results.json").read_text())
for case in result["cases"]:
    print(case["request"])
    print(case["answer"])
    print(f"Receiver: {case['receiver_prefill_and_generation_seconds']:.3f}s")
    print(f"Contains expected text: {case['contains_expected_text']}")

## Optional: validate vLLM

Runs only for the pins listed in `VLLM_PINS`. Each pin gets its own virtual
environment, and each of three repetitions uses a fresh process. The journey
in `scripts/validate_vllm.py` covers token and continuous capture, reasoning
options, native-token parity and a bound-agent return handoff. Raw token IDs,
synchronized timings, peak memory and package versions are saved per run.
Timings describe this workload, not general serving throughput.

In [ ]:
import statistics

REPEATS = 3


def install(pin: str) -> Path:
    """Install one pinned stack in an isolated environment."""
    environment = Path("/content/rcc-envs") / pin
    python = environment / "bin/python"
    env = {**os.environ, "PIP_NO_CACHE_DIR": "1"}
    if not python.exists():
        run_command(
            [sys.executable, "-m", "venv", "--without-pip", str(environment)], f"venv-{pin}.log"
        )
    if not (environment / "bin/pip").exists():
        run_command(
            [sys.executable, "-m", "pip", "--python", str(python), "install", "pip"],
            f"pip-{pin}.log",
            env,
        )
    requirement = "vllm.txt" if pin == "0.11.1" else "vllm-unstable.txt"
    run_command(
        [
            str(python),
            "-m",
            "pip",
            "install",
            "-r",
            str(SOURCE / "requirements" / requirement),
            str(SOURCE),
        ],
        f"install-{pin}.log",
        env,
    )
    run_command([str(python), "-m", "pip", "check"], f"dependencies-{pin}.log")
    run_command([str(python), "-m", "pip", "freeze"], f"packages-{pin}.txt")
    return python


MEASURE = r"""import asyncio
import importlib.metadata
import json
import os
import sys
import time
from pathlib import Path

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"] = "1"
import torch
import rcc
from vllm import LLM

folder = Path(sys.argv[1])
pin = sys.argv[2]
config = json.loads((folder.parent / "config.json").read_text())
folder.mkdir()
report = {
    "complete": False,
    "config": config,
    "gpu": torch.cuda.get_device_name(),
    "versions": {
        p: importlib.metadata.version(p) for p in ("vllm", "torch", "transformers", "rclc")
    },
    "calls": [],
}


def save():
    report["peak_allocated_bytes"] = torch.cuda.max_memory_allocated()
    report["peak_reserved_bytes"] = torch.cuda.max_memory_reserved()
    (folder / "result.json").write_text(json.dumps(report, indent=2) + "\n")


def measured(fn, name):
    def call(*args, **kwargs):
        torch.cuda.synchronize()
        started = time.perf_counter()
        output = fn(*args, **kwargs)
        torch.cuda.synchronize()
        entry = {"phase": name, "seconds": time.perf_counter() - started}
        if name == "generate":
            prompts = kwargs.get("prompts", args[1] if len(args) > 1 else [])
            if isinstance(prompts, dict):
                prompts = [prompts]
            entry["input_positions"] = [
                len(p.get("prompt_embeds", p.get("prompt_token_ids", []))) for p in prompts
            ]
            entry["output_ids"] = [o.outputs[0].token_ids for o in output]
        report["calls"].append(entry)
        save()
        return output

    return call


LLM.generate = measured(LLM.generate, "generate")
async def measured_transfer(*args, **kwargs):
    torch.cuda.synchronize()
    started = time.perf_counter()
    try:
        return await native_transfer(*args, **kwargs)
    finally:
        torch.cuda.synchronize()
        report["calls"].append({"phase": "transfer", "seconds": time.perf_counter() - started})
        save()


native_transfer = rcc.transfer
rcc.transfer = measured_transfer
from rcc import vllm

vllm.prefill_state = measured(vllm.prefill_state, "capture")
from scripts.validate_vllm import run

save()
try:
    tokens = asyncio.run(run(
        allow_unstable=pin == "0.26.0", model=config["model"], revision=config["revision"]
    ))
    assert len(tokens) == 68 and all(len(t) == 4 for t in tokens)
    report["output_ids"] = tokens
    report["complete"] = True
finally:
    save()
print(
    json.dumps(
        {
            "complete": report["complete"],
            "calls": len(report["calls"]),
            "peak_allocated_bytes": report["peak_allocated_bytes"],
        }
    )
)
"""
(RUN / "measure.py").write_text(MEASURE)


def validate(pin: str) -> None:
    """Repeat the GPU journey in fresh processes."""
    python = install(pin)
    env = {**os.environ, "PIP_NO_CACHE_DIR": "1"}
    env["PATH"] = str(python.parent) + os.pathsep + env["PATH"]
    env["PYTHONPATH"] = str(SOURCE) + os.pathsep + str(SOURCE / "src")
    for repeat in range(REPEATS):
        name = f"vllm-{pin}-{repeat + 1}"
        run_command(
            [str(python), "-u", str(RUN / "measure.py"), str(RUN / name), pin], f"{name}.log", env
        )
        result = json.loads((RUN / name / "result.json").read_text())
        print(
            name,
            "complete:",
            result["complete"],
            "peak GiB:",
            round(result["peak_allocated_bytes"] / 1024**3, 2),
        )


for pin in VLLM_PINS:
    validate(pin)
    results = [json.loads(p.read_text()) for p in sorted(RUN.glob(f"vllm-{pin}-*/result.json"))]
    complete = [r for r in results if r["complete"]]
    print(pin, "complete:", len(complete), "/", REPEATS)
    for phase in ("capture", "transfer", "generate"):
        totals = [sum(c["seconds"] for c in r["calls"] if c["phase"] == phase) for r in complete]
        if totals:
            print(" ", phase, "median total seconds:", round(statistics.median(totals), 3))
    if complete:
        peak = max(r["peak_allocated_bytes"] for r in complete) / 1024**3
        print("  peak allocated GiB:", round(peak, 2))

## Download results

Run this even if a step fails. The ZIP includes the source, configuration,
environment, executed scripts, logs and any completed results. Drive also keeps
the files. No model weights or credentials are included. Afterwards, choose
**Runtime > Disconnect and delete runtime** to release the GPU.

In [ ]:
BUNDLE = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
print(f"Saved: {BUNDLE}")
if not TEST_MODE:
    files.download(str(BUNDLE))